# Agentic AI System — Warehouse Intelligence Assistant (WIA)

**Author:** Sébastien Bodrero
**Programme:** Woolf University / Udacity MSc in Artificial Intelligence
**Module:** Agentic AI Systems (Module 6)
**Date:** April 2026

---

This notebook implements a single-agent system called the **Warehouse Intelligence Assistant (WIA)** — an LLM-powered agent that helps warehouse operations managers make real-time fleet coordination decisions. The agent uses the Claude API for reasoning, calls simulated warehouse tools, maintains a decision log in memory, and applies explicit safeguards before issuing any routing recommendation.

**Notebook structure:**
1. [Task 1 — Agentic Task and System Scope](#task1)
2. [Task 2 — Agent Architecture](#task2)
3. [Task 3 — Implementation](#task3)
4. [Task 4 — Execution and Observed Behavior](#task4)
5. [Task 5 — Summary](#task5)
6. [Task 6 — Report Reference](#task6)
7. [Task 7 — Requirements](#task7)

---
## Setup — Imports and Configuration

In [1]:
import os
import json
import datetime
from collections import deque
from typing import Any

import anthropic

print(f"anthropic SDK version : {anthropic.__version__}")
print(f"Notebook executed at  : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

anthropic SDK version : 0.90.0
Notebook executed at  : 2026-04-19 18:14:13


<a id='task1'></a>
---
## Task 1 — Agentic Task and System Scope

### What the agent does

The **Warehouse Intelligence Assistant (WIA)** is a conversational agent that takes natural-language requests from a warehouse operations manager and translates them into concrete, safe fleet-coordination actions. A typical request might be:

> *"Route AGV-3 to pick zone C — what's the current inventory there and is the path clear?"*

The agent gathers context via tools (inventory levels, vehicle status, safe pathfinding), reasons about the best course of action, and produces a recommendation or escalates to a human operator when appropriate.

### Why an agentic approach is appropriate

A static classifier or rule engine cannot handle the **open-ended, multi-step nature** of warehouse coordination queries. The agent must:
- Decide *which* tools to call and *in what order* based on the query
- Integrate information across multiple tool results before reasoning
- Adapt its strategy when a tool returns unexpected results (e.g., zone blocked)
- Apply safety constraints that depend on runtime state, not static rules

This requires an LLM-driven reasoning loop — the hallmark of agentic design.

### Scope and boundaries

| In scope | Out of scope |
|---|---|
| Routing recommendations for named AGVs | Physically issuing commands to real hardware |
| Inventory level checks per zone | Persistent inventory database updates |
| Safe-path computation (BFS on a grid) | Full warehouse digital twin / simulation |
| Human escalation when safety zones are threatened | Multi-agent coordination between WIA instances |
| Decision logging for audit trail | Long-term persistent memory across sessions |

### Design type

**Single-agent** — one WIA instance handles one manager conversation. The agent is not persistent across sessions; each notebook execution is a fresh session. This scope is appropriate for the academic context and mirrors real-world "shift assistant" deployments where state is reset at shift change.

In [2]:
# ── Configuration constants ────────────────────────────────────────────────

# Zones where human operators are present; routing through these requires
# explicit escalation before the agent may recommend a path.
HUMAN_SAFETY_ZONES = {"H1", "H2", "H3", "MAIN_AISLE"}

# Maximum number of tool-call iterations per user turn (prevents runaway loops)
MAX_TOOL_ITERATIONS = 6

# Number of past decisions kept in the rolling decision log
MEMORY_WINDOW = 10

# Claude model to use for the agent's LLM backbone
MODEL_ID = "claude-haiku-4-5-20251001"   # fast & cost-efficient for this demo

print("Configuration loaded.")
print(f"  Safety zones   : {HUMAN_SAFETY_ZONES}")
print(f"  Max iterations : {MAX_TOOL_ITERATIONS}")
print(f"  Memory window  : {MEMORY_WINDOW}")
print(f"  LLM model      : {MODEL_ID}")


Configuration loaded.
  Safety zones   : {'MAIN_AISLE', 'H1', 'H3', 'H2'}
  Max iterations : 6
  Memory window  : 10
  LLM model      : claude-haiku-4-5-20251001


<a id='task2'></a>
---
## Task 2 — Agent Architecture

### Component overview

```
┌─────────────────────────────────────────────────────────────────┐
│                  Warehouse Intelligence Assistant                │
│                                                                 │
│  ┌───────────────┐    ┌──────────────────────────────────────┐  │
│  │  AgentMemory  │◄───│            WarehouseAgent            │  │
│  │               │    │                                      │  │
│  │ • conversation│    │  • System prompt (persona)           │  │
│  │   history     │    │  • run(user_query) → reasoning loop  │  │
│  │ • decision    │    │  • _dispatch_tool(name, args)        │  │
│  │   log         │    │  • _safety_check(tool, args)         │  │
│  └───────────────┘    └──────────────┬───────────────────────┘  │
│                                      │  tool_use / tool_result  │
│                       ┌──────────────▼───────────────────────┐  │
│                       │          WarehouseTools               │  │
│                       │                                      │  │
│                       │  check_inventory(zone_id)            │  │
│                       │  get_agent_status(agent_id)          │  │
│                       │  find_safe_path(start, end, blocked) │  │
│                       │  escalate_to_human(reason, urgency)  │  │
│                       └──────────────────────────────────────┘  │
│                                      │  Claude API              │
│                       ┌──────────────▼───────────────────────┐  │
│                       │         Anthropic Claude API          │  │
│                       │    (claude-haiku-4-5-20251001)        │  │
│                       └──────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────────────┘
```

### Reasoning loop (ReAct-style)

```
User query
    │
    ▼
[1] Build messages (system prompt + memory + query)
    │
    ▼
[2] Call Claude API with tool definitions
    │
    ├── stop_reason = "end_turn"  → return final text answer
    │
    └── stop_reason = "tool_use"
            │
            ▼
        [3] Safety check (is this tool call safe?)
                │
                ├── BLOCKED → auto-escalate, append warning, continue
                │
                └── OK → dispatch tool, get result
                            │
                            ▼
                        [4] Append tool_result to messages
                            │
                            ▼
                        [5] goto [2]  (until end_turn or MAX_TOOL_ITERATIONS)
```

### Design choices and tradeoffs

| Choice | Rationale | Tradeoff |
|---|---|---|
| Single-agent (not multi-agent) | Simpler credit assignment; sufficient for this scope | Cannot parallelise sub-tasks across specialised agents |
| Simulated tools (not real APIs) | No external infrastructure dependency; fully reproducible | Does not reflect real warehouse API latency or failures |
| In-session memory only | No persistent state complicates reproducibility and safety | Agent forgets decisions between notebook restarts |
| Claude Haiku (not Sonnet/Opus) | Faster, cheaper for iterative testing | Slightly less nuanced reasoning on ambiguous queries |
| Safety check before dispatch | Fail-safe: block unsafe paths *before* tool execution | Adds latency; may over-block edge cases |
| BFS pathfinding (not A*) | Simple, predictable, no heuristic tuning | Suboptimal for large grids; A* would be better in production |

<a id='task3'></a>
---
## Task 3 — Implementation

In [3]:
# ── Simulated warehouse state ──────────────────────────────────────────────
# In production this would come from a live WMS (Warehouse Management System).
# Here we use static Python dicts for full reproducibility.

WAREHOUSE_INVENTORY = {
    "A": {"item": "SKU-1001 (Electronics)",   "units": 142, "status": "normal"},
    "B": {"item": "SKU-2034 (Textiles)",       "units": 0,   "status": "empty"},
    "C": {"item": "SKU-3087 (Automotive)",     "units": 58,  "status": "low"},
    "D": {"item": "SKU-4112 (Pharmaceuticals)","units": 310, "status": "normal"},
    "E": {"item": "SKU-5200 (Food & Beverage)","units": 75,  "status": "normal"},
}

FLEET_STATUS = {
    "AGV-1": {"type": "forklift",     "location": "A",        "battery": 87,  "status": "idle"},
    "AGV-2": {"type": "conveyor-bot", "location": "CHARGING", "battery": 12,  "status": "charging"},
    "AGV-3": {"type": "forklift",     "location": "D",        "battery": 65,  "status": "idle"},
    "AGV-4": {"type": "drone",        "location": "H2",       "battery": 44,  "status": "busy"},
    "AGV-5": {"type": "conveyor-bot", "location": "E",        "battery": 91,  "status": "idle"},
}

# Simple 5×5 grid for BFS path-finding.  Zones are nodes; adjacency is direct.
ZONE_GRAPH = {
    "A":        ["B", "MAIN_AISLE"],
    "B":        ["A", "C", "H1"],
    "C":        ["B", "D", "MAIN_AISLE"],
    "D":        ["C", "E", "H2"],
    "E":        ["D", "MAIN_AISLE"],
    "H1":       ["B", "H2"],
    "H2":       ["H1", "H3", "D"],
    "H3":       ["H2", "MAIN_AISLE"],
    "MAIN_AISLE":["A", "C", "E", "H3", "CHARGING"],
    "CHARGING": ["MAIN_AISLE"],
}

print("Warehouse state loaded.")

Warehouse state loaded.


In [4]:
class WarehouseTools:
    """
    Simulated warehouse tool layer.

    Each method corresponds to one tool the agent may invoke.  All methods
    return a dict so results can be serialised directly to JSON for the
    Claude tool_result message.
    """

    def check_inventory(self, zone_id: str) -> dict:
        """
        Return current inventory levels for a warehouse zone.

        Parameters
        ----------
        zone_id : str
            Zone identifier (e.g. 'A', 'B', 'C', 'D', 'E').

        Returns
        -------
        dict
            Keys: zone, item, units, status.  'error' key present on failure.
        """
        zone_id = zone_id.upper()
        if zone_id not in WAREHOUSE_INVENTORY:
            return {"error": f"Zone '{zone_id}' not found in inventory system."}
        data = WAREHOUSE_INVENTORY[zone_id].copy()
        data["zone"] = zone_id
        return data

    def get_agent_status(self, agent_id: str) -> dict:
        """
        Return current status of a fleet vehicle (AGV, drone, conveyor-bot).

        Parameters
        ----------
        agent_id : str
            Vehicle identifier (e.g. 'AGV-1', 'AGV-3').

        Returns
        -------
        dict
            Keys: agent_id, type, location, battery, status.  'error' if unknown.
        """
        if agent_id not in FLEET_STATUS:
            return {"error": f"Vehicle '{agent_id}' not registered in fleet."}
        data = FLEET_STATUS[agent_id].copy()
        data["agent_id"] = agent_id
        return data

    def find_safe_path(self, start: str, end: str, blocked_zones: list[str]) -> dict:
        """
        Find a path from start to end in the warehouse zone graph using BFS,
        avoiding any explicitly blocked zones.

        Parameters
        ----------
        start : str
            Starting zone identifier.
        end : str
            Destination zone identifier.
        blocked_zones : list[str]
            Zones to exclude from the path (e.g. zones under maintenance).

        Returns
        -------
        dict
            Keys: path (list of zones), hops (int).  'error' if no path found.

        Notes
        -----
        BFS guarantees the shortest path in terms of number of zone transitions.
        It does not account for travel time or congestion.
        """
        start = start.upper()
        end   = end.upper()
        blocked = {z.upper() for z in (blocked_zones or [])}

        if start not in ZONE_GRAPH:
            return {"error": f"Unknown start zone '{start}'."}
        if end not in ZONE_GRAPH:
            return {"error": f"Unknown destination zone '{end}'."}

        # BFS
        queue   = deque([[start]])
        visited = {start}
        while queue:
            path = queue.popleft()
            node = path[-1]
            if node == end:
                return {"path": path, "hops": len(path) - 1}
            for neighbour in ZONE_GRAPH.get(node, []):
                if neighbour not in visited and neighbour not in blocked:
                    visited.add(neighbour)
                    queue.append(path + [neighbour])

        return {"error": f"No path from '{start}' to '{end}' avoiding {blocked}."}

    def escalate_to_human(self, reason: str, urgency: str) -> dict:
        """
        Log an escalation event and notify the human operator.

        In a production system this would send a push notification or page the
        shift supervisor.  Here we log the event and return a confirmation.

        Parameters
        ----------
        reason : str
            Human-readable description of why escalation is needed.
        urgency : str
            One of: 'low', 'medium', 'high', 'critical'.

        Returns
        -------
        dict
            Escalation record with timestamp and ticket_id.
        """
        ticket_id = f"ESC-{datetime.datetime.now().strftime('%H%M%S')}"
        record = {
            "ticket_id"  : ticket_id,
            "timestamp"  : datetime.datetime.now().isoformat(),
            "reason"     : reason,
            "urgency"    : urgency,
            "status"     : "OPEN",
            "message"    : (
                f"[{urgency.upper()}] Human operator notified — ticket {ticket_id}. "
                "Awaiting operator acknowledgement before proceeding."
            ),
        }
        print(f"  ⚠  ESCALATION RAISED: {ticket_id} [{urgency.upper()}] — {reason}")
        return record


tools_instance = WarehouseTools()
print("WarehouseTools initialised.")

WarehouseTools initialised.
